# 1. Data Preprocessing & Validation

This notebook implements 'Poetry-Safe' cleaning, language verification, label validation, and mandatory statistics generation.

In [ ]:
import pandas as pd
import re
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style='whitegrid')

## 1.1 Load Data

In [ ]:
file_path = 'round1/Primary_Emotions.xlsx'
df = pd.read_excel(file_path)
print(f"Loaded {len(df)} records.")
df.head()

## 1.4 Minimal Text Cleaning (POETRY-SAFE)
- Strip whitespace
- Normalize newlines
- Standardize punctuation spacing
- **NO** stemming, lemmatization, stopword removal, or lowercasing.

In [ ]:
def clean_poem(text):
    if not isinstance(text, str):
        return str(text)
    
    # Strip leading/trailing whitespace
    text = text.strip()
    
    # Normalize repeated newlines
    text = re.sub(r'\n+', '\n', text)
    
    # Standardize punctuation spacing (remove space before punctuation force space after if missing)
    text = re.sub(r'\s+([,.;!?])', r'\1', text)
    
    return text

df['cleaned_poem'] = df['Poem'].apply(clean_poem)
print("Text cleaning applied. Sample:")
print(df['cleaned_poem'].iloc[0])

## 1.5 Language Verification
Checking script consistency with 'Source' column.

In [ ]:
def check_script_match(text, language):
    # Unicode ranges for scripts
    scripts = {
        'Hindi': r'[\u0900-\u097F]',
        'Tamil': r'[\u0B80-\u0BFF]',
        'Telugu': r'[\u0C00-\u0C7F]',
        'Malayalam': r'[\u0D00-\u0D7F]',
        'Kannada': r'[\u0C80-\u0CFF]',
        'Bengali': r'[\u0980-\u09FF]',
        'English': r'[a-zA-Z]'
    }
    
    if language not in scripts:
        return True # Skip unknown languages
        
    pattern = scripts[language]
    if re.search(pattern, text):
        return True
    return False

df['script_match'] = df.apply(lambda x: check_script_match(x['cleaned_poem'], x['Source']), axis=1)
mismatches = df[~df['script_match']]
print(f"Language Mismatches: {len(mismatches)}")
if not mismatches.empty:
    display(mismatches[['Poem', 'Source']])

## 1.6 Label Validation
Ensuring exactly 46 unique labels.

In [ ]:
unique_labels = df['Primary'].unique()
label_count = len(unique_labels)
print(f"Unique Labels: {label_count}")

expected_count = 46
if label_count == expected_count:
    print("✅ Label count matches expected (46).")
else:
    print(f"⚠️ Mismatch! Expected {expected_count}, found {label_count}.")
    
label_counts = df['Primary'].value_counts()
print(label_counts)

## 1.7 Dataset Statistics (MANDATORY)

In [ ]:
# Total poems
print(f"Total Poems: {len(df)}")

# Poems per Language
print("\nPoems per Language:")
print(df['Source'].value_counts())

# Average Length (Words)
df['word_count'] = df['cleaned_poem'].apply(lambda x: len(x.split()))
avg_len = df['word_count'].mean()
print(f"\nAverage Poem Length (words): {avg_len:.2f}")

# Length Distribution Plot
plt.figure(figsize=(10, 5))
sns.histplot(df['word_count'], bins=30, kde=True)
plt.title('Distribution of Poem Lengths (Words)')
plt.xlabel('Word Count')
plt.show()

In [ ]:
# Save processed data for next steps
df.to_excel('round1/Primary_Emotions_Processed.xlsx', index=False)
print("Processed data saved to 'round1/Primary_Emotions_Processed.xlsx'")